# Visualisation 2 — Effet régional sur les prénoms français

Réponse aux questions Q1, Q2, Q3 via un dashboard interactif à 4 panneaux :

| Panneau | Vue | Questions |
|---|---|---|
| **A — Carte de distinctivité** | Choroplèthe (divergence de Jensen-Shannon) | Q1 |
| **B — Top prénoms locaux** | Bar chart (lift, prénoms fréquents) | Q2 |
| **C — Heatmap clusterisée** | Départements × prénoms, trié par clustering de Ward | Q1 + Q2 + Q3 |
| **D — Explorateur de prénom** | Carte divergente pour un prénom choisi | Q2 + Q3 |




## 1. Imports & configuration


In [26]:
import altair as alt

# CELL 1 — Imports & configuration
import json
import numpy as np
import pandas as pd
import altair as alt
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

alt.data_transformers.disable_max_rows()

# ---- CONFIGURATION (modifiable : relancer le notebook après changement) ----
YEAR_MIN, YEAR_MAX = 1970, 2020   # fenêtre d'analyse
ALPHA            = 1000   # lissage bayésien de la distinctivité (vers la distribution nationale)
UNIV_MIN_NAT     = 1000   # un prénom entre dans l'« univers » de comparaison s'il est >= ce nb de naissances nat.
TOP_NAT_K        = 25     # nb de prénoms nationaux en colonnes de la heatmap
LOCAL_K          = 15     # nb de prénoms « régionaux » supplémentaires en colonnes
TOP_LIFT_K       = 10     # nb de barres dans le Panneau B
LOC_MIN_SHARE    = 0.0015 # >= 0,15 % des naissances locales (plancher « caractéristique »)
LOC_MIN_NAT_FULL = 800    # fréquence nat. mini sur la fenêtre complète (sélection des colonnes)
LOC_MIN_NAT_DEC  = 300    # fréquence nat. mini par décennie (Panneau B)
LOC_MIN_COUNT    = 30     # garde-fou petit nombre

print(f"pandas {pd.__version__} | numpy {np.__version__} | altair {alt.__version__}")
print(f"Fenêtre d'analyse : {YEAR_MIN}–{YEAR_MAX}")

pandas 2.3.3 | numpy 2.1.3 | altair 5.5.0
Fenêtre d'analyse : 1970–2020


## 2. Chargement et nettoyage des données

### Source
- `dpt2020.csv` : 1 ligne par triplet (sexe, prénom, année, département) avec le nombre de naissances.
- `departements-version-simplifiee.geojson` : tracés simplifiés des départements métropolitains.

### Nettoyage
- On retire les lignes `_PRENOMS_RARES` (agrégation INSEE des prénoms trop rares) et `dpt == 'XX'` (département non renseigné).
- On agrège les sexes (F+M) - le genre est géré à travers la visu 3
- On capitalise proprement les prénoms pour la lisibilité.


In [27]:
# CELL 2 — Chargement, nettoyage
names_raw = pd.read_csv("../../dpt2020.csv", sep=";", dtype={"dpt": str})
keep = (names_raw["preusuel"] != "_PRENOMS_RARES") & (names_raw["dpt"] != "XX") & (names_raw["annais"] != "XXXX")
names = names_raw.loc[keep].copy()
names["annais"] = names["annais"].astype(int)
names["nombre"] = names["nombre"].astype(int)
names["preusuel"] = names["preusuel"].str.capitalize()

with open("../../departements-version-simplifiee.geojson", encoding="utf-8") as f:
    depts_geojson = json.load(f)
geo_codes = {ft["properties"]["code"] for ft in depts_geojson["features"]}
dpt_label = {ft["properties"]["code"]: ft["properties"]["nom"] for ft in depts_geojson["features"]}
dpt_label["20"] = "Corse" 


analysis_dpts = (geo_codes - {"2A", "2B"}) | {"20"}
names = names[names["dpt"].isin(analysis_dpts)].copy()

def decade_of(y): return min((y // 10) * 10, 2010)   # 2020 replié dans les années 2010
names["decade"] = names["annais"].map(decade_of)
window = names[names["annais"].between(YEAR_MIN, YEAR_MAX)].copy()
DECADES = sorted(window["decade"].unique().tolist())
print(f"Lignes (fenêtre) : {len(window):,} | départements : {window['dpt'].nunique()} | décennies : {DECADES}")

Lignes (fenêtre) : 2,106,717 | départements : 95 | décennies : [1970, 1980, 1990, 2000, 2010]


In [28]:
# CELL 3 — Fonctions centrales (lift, distinctivité, clustering)
def aggregate(df):
    """(dpt, prénom) -> count_local, share_local, share_nat, lift, log_lift sur la sous-période df."""
    c = df.groupby(["dpt", "preusuel"], as_index=False)["nombre"].sum().rename(columns={"nombre": "count_local"})
    tot_nat = c["count_local"].sum()
    nat = c.groupby("preusuel", as_index=False)["count_local"].sum().rename(columns={"count_local": "count_nat"})
    nat["share_nat"] = nat["count_nat"] / tot_nat
    tot = c.groupby("dpt", as_index=False)["count_local"].sum().rename(columns={"count_local": "total_dpt"})
    e = c.merge(tot, on="dpt").merge(nat, on="preusuel")
    e["share_local"] = e["count_local"] / e["total_dpt"]
    e["lift"] = e["share_local"] / e["share_nat"]
    e["log_lift"] = np.log2(e["lift"])
    return e

def _smoothed_profiles(df, univ, alpha=ALPHA):
    """(depts, P_lissé, q_national) sur l'univers de prénoms `univ`."""
    e = aggregate(df)
    sub = e[e["preusuel"].isin(univ)]
    nat = sub.groupby("preusuel")["count_local"].sum()
    q = (nat / nat.sum()).reindex(univ).fillna(0).values
    M = (sub.pivot_table(index="dpt", columns="preusuel", values="count_local", fill_value=0)
            .reindex(columns=univ, fill_value=0))
    depts = M.index.tolist()
    counts = M.values.astype(float); tot = counts.sum(1, keepdims=True)
    P = (counts + alpha * q) / (tot + alpha)          # lissage vers le national
    return depts, P, q

def _js(p, q):
    mm = 0.5 * (p + q)
    kl = lambda a, b: np.sum(a[a > 0] * np.log2(a[a > 0] / b[a > 0]))
    return 0.5 * kl(p, mm) + 0.5 * kl(q, mm)

def distinctiveness(df, univ, alpha=ALPHA):
    depts, P, q = _smoothed_profiles(df, univ, alpha)
    vals = np.array([_js(P[i], q) for i in range(len(depts))])
    return pd.DataFrame({"dpt": depts, "distinctiveness": vals})

def clustering_order(df, univ, alpha=ALPHA):
    depts, P, _ = _smoothed_profiles(df, univ, alpha)
    n = len(depts); D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = np.sqrt(max(_js(P[i], P[j]), 0.0))
    Z = linkage(squareform(D, checks=False), method="ward")
    return [depts[i] for i in leaves_list(Z)]

## 3. Fenêtre temporelle

Analyse sur **1970–2020** (5 décennies). Point de départ : 1970, après la réforme
administrative de 1968 qui a créé les départements 92/93/94/95 — avant cette date
ils n'ont aucune donnée et apparaîtraient en gris sur la carte.
Paramétrable via `YEAR_MIN` / `YEAR_MAX` en cellule 1.

In [29]:
# CELL 4 
enr = aggregate(window)                                    # lift de tous les (dpt, prénom)
UNIV = enr.groupby("preusuel")["count_nat"].first()
UNIV = UNIV[UNIV >= UNIV_MIN_NAT].index.tolist()           # univers de prénoms
dpt_order = clustering_order(window, UNIV)                 # ordre des lignes de la heatmap
distinct_full = distinctiveness(window, UNIV)              # distinctivité de référence

dn_df = pd.DataFrame({"dpt": list(dpt_label), "nom": list(dpt_label.values())})
print(f"Univers de prénoms (>= {UNIV_MIN_NAT} nat.) : {len(UNIV)}")
print("TOP 8 départements les plus distinctifs (fenêtre complète) :")
print(distinct_full.merge(dn_df, on="dpt").nlargest(8, "distinctiveness")[["dpt", "nom", "distinctiveness"]].to_string(index=False))

Univers de prénoms (>= 1000 nat.) : 1702
TOP 8 départements les plus distinctifs (fenêtre complète) :
dpt         nom  distinctiveness
 20       Corse         0.152163
 23      Creuse         0.143148
 48      Lozère         0.135650
 15      Cantal         0.109568
 32        Gers         0.106514
 46         Lot         0.105345
 43 Haute-Loire         0.100515
 09      Ariège         0.094894


## 4. Métriques centrales

- **Lift** : `share_local / share_national`. Mesure la sur- ou sous-représentation
  locale d'un prénom. Calculé uniquement sur les prénoms dépassant un seuil de
  fréquence nationale pour éviter les artefacts de rareté.
- **Distinctivité** : divergence de Jensen-Shannon (lissée) entre la distribution
  locale et nationale. Utilisée pour colorer la carte A.
- **Clustering** : Ward sur distances Jensen-Shannon entre départements. Détermine
  l'ordre des lignes de la heatmap C — les blocs culturels émergent sans prior géographique.

In [30]:
# CELL 5 — Sélection des colonnes de la heatmap (nationaux + régionaux)
nat_full = enr.groupby("preusuel", as_index=False)["count_nat"].first()
top_national = nat_full.nlargest(TOP_NAT_K, "count_nat")["preusuel"].tolist()

cand = enr[(enr["count_nat"] >= LOC_MIN_NAT_FULL) & (enr["count_local"] >= 100) & (enr["share_local"] >= LOC_MIN_SHARE)]
maxlift = cand.groupby("preusuel", as_index=False)["lift"].max().rename(columns={"lift": "max_lift"})
local_names = (maxlift[~maxlift["preusuel"].isin(top_national)]
               .nlargest(LOCAL_K, "max_lift")["preusuel"].tolist())

selected_names = top_national + local_names
print(f"NATIONAUX ({len(top_national)}) : {top_national}")
print(f"RÉGIONAUX ({len(local_names)}) : {local_names}")

NATIONAUX (25) : ['Nicolas', 'Sébastien', 'Julien', 'David', 'Thomas', 'Alexandre', 'Christophe', 'Stéphanie', 'Céline', 'Stéphane', 'Camille', 'Maxime', 'Marie', 'Guillaume', 'Frédéric', 'Julie', 'Romain', 'Sandrine', 'Vincent', 'Sophie', 'Aurélie', 'Nathalie', 'Lucas', 'Antoine', 'Léa']
RÉGIONAUX (15) : ['Francesca', 'Iban', 'Vanina', 'Pierre-marie', 'Marc-antoine', 'Ange', 'Séréna', 'Chrystele', 'Katell', 'Pierre-antoine', 'Rozenn', 'Joan', 'Livia', 'Gwendal', 'Maylis']


In [31]:
# CELL 6 — Pré-calcul par décennie
def add_corsica_for_map(df):
    """Duplique la Corse '20' -> '2A' et '2B' pour la jointure avec le GeoJSON."""
    cor = df[df["dpt"] == "20"]
    extra = pd.concat([cor.assign(dpt="2A"), cor.assign(dpt="2B")], ignore_index=True)
    return pd.concat([df[df["dpt"] != "20"], extra], ignore_index=True)

heat_rows, distinct_rows, panelB_rows = [], [], []
for dec in DECADES:
    dfd = window[window["decade"] == dec]
    ed = aggregate(dfd)
    h = ed[ed["preusuel"].isin(selected_names)].copy(); h["decade"] = dec
    heat_rows.append(h)
    dd = distinctiveness(dfd, UNIV); dd["decade"] = dec
    distinct_rows.append(dd)
    b = ed[(ed["share_local"] >= LOC_MIN_SHARE) & (ed["count_nat"] >= LOC_MIN_NAT_DEC) & (ed["count_local"] >= LOC_MIN_COUNT)].copy()
    b["rk"] = b.groupby("dpt")["lift"].rank(ascending=False, method="first")
    b = b[b["rk"] <= TOP_LIFT_K]; b["decade"] = dec
    panelB_rows.append(b)

heat = pd.concat(heat_rows, ignore_index=True)
heat["log_lift_clip"] = heat["log_lift"].clip(-3, 3)
heat["dpt_name"] = heat["dpt"].map(dpt_label)
distinct_dec = pd.concat(distinct_rows, ignore_index=True); distinct_dec["dpt_name"] = distinct_dec["dpt"].map(dpt_label)
panelB = pd.concat(panelB_rows, ignore_index=True); panelB["dpt_name"] = panelB["dpt"].map(dpt_label)


distinct_dec_map = add_corsica_for_map(distinct_dec)
heat_map = add_corsica_for_map(heat)

print(f"heat {heat.shape} | distinct_dec {distinct_dec.shape} | panelB {panelB.shape}")
print(f"heat_map {heat_map.shape} | distinct_dec_map {distinct_dec_map.shape}")

heat (12392, 12) | distinct_dec (475, 4) | panelB (4750, 12)
heat_map (12540, 12) | distinct_dec_map (480, 4)


In [32]:
# CELL 7 — Validation rapide
print("ORDRE DU CLUSTERING (lecture des lignes de la heatmap) :")
for k in range(0, len(dpt_order), 6):
    print("  " + " | ".join(f"{d}·{dpt_label.get(d, d)[:10]}" for d in dpt_order[k:k+6]))

print("\nPrénoms locaux par décennie (Panneau B) :")
for code, lab in [("29", "Finistère"), ("64", "Pyr-Atl"), ("20", "Corse"), ("93", "Seine-St-Denis")]:
    print(f"  {lab} :")
    for dec in DECADES:
        sub = panelB[(panelB["dpt"] == code) & (panelB["decade"] == dec)].nlargest(5, "lift")
        print(f"    {dec}s : " + ", ".join(f"{r.preusuel} (×{r.lift:.0f})" for r in sub.itertuples()))

ORDRE DU CLUSTERING (lecture des lignes de la heatmap) :
  30·Gard | 34·Hérault | 84·Vaucluse | 13·Bouches-du | 06·Alpes-Mari | 83·Var
  31·Haute-Garo | 33·Gironde | 38·Isère | 69·Rhône | 26·Drôme | 42·Loire
  75·Paris | 78·Yvelines | 92·Hauts-de-S | 93·Seine-Sain | 77·Seine-et-M | 91·Essonne
  94·Val-de-Mar | 95·Val-d'Oise | 23·Creuse | 48·Lozère | 32·Gers | 46·Lot
  15·Cantal | 43·Haute-Loir | 20·Corse | 19·Corrèze | 03·Allier | 16·Charente
  24·Dordogne | 87·Haute-Vien | 12·Aveyron | 05·Hautes-Alp | 07·Ardèche | 66·Pyrénées-O
  11·Aude | 81·Tarn | 47·Lot-et-Gar | 82·Tarn-et-Ga | 64·Pyrénées-A | 40·Landes
  65·Hautes-Pyr | 04·Alpes-de-H | 09·Ariège | 53·Mayenne | 50·Manche | 61·Orne
  79·Deux-Sèvre | 17·Charente-M | 86·Vienne | 85·Vendée | 49·Maine-et-L | 14·Calvados
  72·Sarthe | 35·Ille-et-Vi | 44·Loire-Atla | 29·Finistère | 22·Côtes-d'Ar | 56·Morbihan
  36·Indre | 58·Nièvre | 70·Haute-Saôn | 52·Haute-Marn | 55·Meuse | 08·Ardennes
  02·Aisne | 27·Eure | 10·Aube | 89·Yonne | 18·Cher

## 5. Visualisation


In [33]:
# CELL 8 — Paramètres + Carte A


decade_param = alt.param(
    name="decade_val",
    value=2010,
    bind=alt.binding_range(
        min=min(DECADES), max=max(DECADES), step=10, name="Décennie : "
    ),
)


name_param = alt.param(
    name="name_val",
    value=local_names[0],
    bind=alt.binding_select(options=selected_names, name="Prénom : "),
)

dept_sel = alt.selection_point(
    fields=["dpt"], name="dept_sel", on="click", empty=False,
    value=[{"dpt": "29"}],
)

geo_lookup = alt.LookupData(
    data=alt.Data(values=depts_geojson["features"]),
    key="properties.code",
    fields=["type", "geometry"],
)

dmax = float(distinct_dec_map["distinctiveness"].max())

map_distinct = (
    alt.Chart(distinct_dec_map)
    .mark_geoshape()
    .transform_filter("datum.decade == decade_val")
    .transform_lookup(lookup="dpt", from_=geo_lookup)
    .encode(
        color=alt.Color(
            "distinctiveness:Q",
            scale=alt.Scale(scheme="yelloworangered", domain=[0, dmax]),
            legend=alt.Legend(title="Distinctivité"),
        ),
        stroke=alt.condition(dept_sel, alt.value("#111"), alt.value("#fff")),
        strokeWidth=alt.condition(dept_sel, alt.value(2.0), alt.value(0.4)),
        tooltip=[
            alt.Tooltip("dpt_name:N",        title="Département"),
            alt.Tooltip("distinctiveness:Q",  title="Distinctivité", format=".3f"),
            alt.Tooltip("decade:O",           title="Décennie"),
        ],
    )
    .project(type="mercator")
    .properties(
        width=420, height=460,
        title=alt.TitleParams(
            "A — Distinctivité des prénoms (Q1)",
            subtitle="Décennie ↑ · Clic sur un département = prénoms caractéristiques ▶",
            anchor="start",
        ),
    )
    
    .add_params(decade_param, name_param, dept_sel)
)

print("cell 8 OK")

cell 8 OK


In [34]:
# CELL 9 — Panneau B

panneau_b = (
    alt.Chart(panelB)
    .transform_filter(dept_sel)
    .transform_filter("datum.decade == decade_val")
    .mark_bar(color="#2a9d8f")
    .encode(
        x=alt.X("lift:Q", title="Lift (× moy. nationale)"),
        y=alt.Y("preusuel:N", sort="-x", title=None),
        tooltip=[
            alt.Tooltip("preusuel:N",    title="Prénom"),
            alt.Tooltip("dpt_name:N",    title="Département"),
            alt.Tooltip("lift:Q",        title="Lift",              format=".1f"),
            alt.Tooltip("count_local:Q", title="Naissances locales", format=","),
            alt.Tooltip("share_local:Q", title="Part locale",        format=".2%"),
            alt.Tooltip("share_nat:Q",   title="Part nationale",     format=".2%"),
        ],
    )
    .properties(
        width=320, height=280,
        title=alt.TitleParams(
            "B — Prénoms caractéristiques (Q2)",
            subtitle="◀ Cliquez un département sur la carte A",
            anchor="start",
        ),
    )
)

print("cell 9 OK — panneau_b défini")

cell 9 OK — panneau_b défini


In [35]:
# CELL 10 — Carte D

base_grey = (
    alt.Chart(alt.Data(values=depts_geojson["features"]))
    .mark_geoshape(fill="#ddd", stroke="white", strokeWidth=0.4)
    .properties(width=420, height=460)
)

color_layer = (
    alt.Chart(heat_map)
    .mark_geoshape()
    .transform_filter("datum.decade == decade_val")
    .transform_filter("datum.preusuel == name_val")
    .transform_lookup(lookup="dpt", from_=geo_lookup)
    .encode(
        color=alt.Color(
            "log_lift_clip:Q",
            scale=alt.Scale(scheme="redblue", reverse=True, domainMid=0, domain=[-3, 3]),
            legend=alt.Legend(title="log₂(lift)"),
        ),
        tooltip=[
            alt.Tooltip("dpt_name:N",    title="Département"),
            alt.Tooltip("preusuel:N",    title="Prénom"),
            alt.Tooltip("lift:Q",        title="Lift",               format=".2f"),
            alt.Tooltip("count_local:Q", title="Naissances locales",  format=","),
            alt.Tooltip("share_local:Q", title="Part locale",         format=".2%"),
            alt.Tooltip("share_nat:Q",   title="Part nationale",      format=".2%"),
        ],
    )
    .properties(width=420, height=460)
)

map_name = (
    alt.layer(base_grey, color_layer)
    .project(type="mercator")
    .properties(
        title=alt.TitleParams(
            "D — Carte d'un prénom (Q2 / Q3)",
            subtitle="Rouge = sur-représenté · Blanc = moy. nationale · Bleu = sous-représenté",
            anchor="start",
        )
    )
    # pas de add_params : decade_param et name_param sont dans le bandeau de contrôle
)

print("cell 10 OK")

cell 10 OK


In [36]:
# CELL 11 — Heatmap
BLOCS = [
    ("Médit. & Rhône-Alpes",  ["30","34","84","13","06","83","31","33","38","69","26","42"]),
    ("Île-de-France",          ["75","78","92","93","77","91","94","95"]),
    ("Massif Central & Corse", ["23","48","32","46","15","43","20","19","03","16"]),
    ("Occitanie & Pyrénées",   ["24","87","12","05","07","66","11","81","47","82","64","40","65","04","09"]),
    ("Grand Ouest",            ["53","50","61","79","17","86","85","49","14","72"]),
    ("Bretagne",               ["35","44","29","22","56"]),
    ("Centre & Nord",          ["36","58","70","52","55","08","02","27","10","89","18","41"]),
    ("Nord-Est",               ["39","88","59","62","80","21","25","51","54","28","45","60","37","76"]),
    ("Alsace-Moselle",         ["57","67","68","90"]),
    ("Alpes & Auvergne",       ["73","74","01","63","71"]),
]
BLOC_COLORS = [
    "#4878d0","#ee854a","#6acc65","#d65f5f","#956cb4",
    "#8c613c","#dc7ec0","#797979","#d5bb67","#82c6e2",
]
HM_W, HM_H = 790, 590

bloc_df_dpt = pd.DataFrame(
    [(d, bname) for bname, depts in BLOCS for d in depts],
    columns=["dpt", "bloc"],
)
missing = set(dpt_order) - set(bloc_df_dpt["dpt"])
if missing: print(f"⚠ depts manquants dans BLOCS : {sorted(missing)}")

strip = (
    alt.Chart(bloc_df_dpt)
    .mark_rect()
    .encode(
        y=alt.Y("dpt:N", sort=dpt_order,
                axis=alt.Axis(labels=False, ticks=False, domain=False, title=None)),
        color=alt.Color(
            "bloc:N",
            scale=alt.Scale(domain=[b[0] for b in BLOCS], range=BLOC_COLORS),
            legend=alt.Legend(title="Région culturelle", orient="bottom", columns=3),
        ),
    )
    .properties(width=15, height=HM_H)
)

heatmap = (
    alt.Chart(heat)
    .mark_rect()
    .transform_filter("datum.decade == decade_val")
    .encode(
        x=alt.X("preusuel:N", sort=selected_names, title="Prénom",
                axis=alt.Axis(labelAngle=-50, labelLimit=90, labelFontSize=9)),
        y=alt.Y("dpt:N", sort=dpt_order,
                title="Département (trié par clustering culturel)",
                axis=alt.Axis(labelFontSize=7, labelLimit=110)),
        color=alt.Color(
            "log_lift_clip:Q",
            scale=alt.Scale(scheme="redblue", reverse=True, domainMid=0),
            legend=alt.Legend(title="log₂(lift)"),
        ),
        stroke=alt.condition(dept_sel, alt.value("#111"), alt.value("#ccc")),
        strokeWidth=alt.condition(dept_sel, alt.value(1.5), alt.value(0)),
        tooltip=[
            alt.Tooltip("dpt_name:N",    title="Département"),
            alt.Tooltip("preusuel:N",    title="Prénom"),
            alt.Tooltip("lift:Q",        title="Lift",              format=".2f"),
            alt.Tooltip("count_local:Q", title="Naissances locales", format=","),
            alt.Tooltip("share_local:Q", title="Part locale",        format=".2%"),
            alt.Tooltip("decade:O",      title="Décennie"),
        ],
    )
    .properties(width=HM_W, height=HM_H,
        title=alt.TitleParams(
            "C — Heatmap département × prénoms  (Q1 + Q2 + Q3)",
            subtitle=[
                "Rouge = sur-représenté  ·  Blanc = moy. nationale  ·  Bleu = sous-représenté",
                "Par LIGNE → Q2  ·  Par COLONNE → Q3  ·  Par BLOC → Q1",
            ],
            anchor="start",
        ),
    )
    
)

full_heatmap = alt.hconcat(strip, heatmap, spacing=0)
print("cell 11 OK")

cell 11 OK


In [37]:
# CELL 12 
import re


top_row = alt.hconcat(map_distinct, panneau_b).resolve_scale(color="independent")

dashboard = (
    alt.vconcat(top_row, map_name, full_heatmap)
    .resolve_scale(color="independent")
    .properties(
        title=alt.TitleParams(
            "Visualisation 2 — Effet régional sur les prénoms français",
            subtitle=f"Données INSEE dpt2020 · {YEAR_MIN}–{YEAR_MAX} · "
                     f"France métropolitaine (95 dép.) · Altair / Vega-Lite",
            anchor="start", fontSize=16,
        )
    )
)

dashboard.save("viz2_regional.html", inline=True)

with open("viz2_regional.html", "r", encoding="utf-8") as f:
    html = f.read()

css = """
<style>
.vega-bindings {
    padding-left: 480px;
    padding-bottom: 10px;
    font-family: sans-serif;
    font-size: 13px;
    border-bottom: 1px solid #ddd;
    margin-bottom: 6px;
}
.vega-bind {
    display: inline-block;
    margin-right: 22px;
    vertical-align: middle;
}
.vega-bind select {
    -webkit-appearance: menulist !important;
    appearance: menulist !important;
    height: 28px !important;
    max-height: 28px !important;
    font-size: 13px;
    min-width: 150px;
    cursor: pointer;
}
.vega-bind input[type=range] {
    width: 160px;
    vertical-align: middle;
}
</style>
"""
html = html.replace("</head>", css + "</head>")


move_js = (
    ".then(function(result){"
    "function fix(){"
    "var ct=document.getElementById('vis');"
    "var bn=ct.querySelector('.vega-bindings');"
    "if(!bn)return;"
    "ct.insertBefore(bn,ct.firstChild);"

    "bn.querySelectorAll('select').forEach(function(orig){"
    "if(orig.dataset.fixed)return;"
    "orig.dataset.fixed='1';"
    "var neo=document.createElement('select');"
    "neo.style.cssText='height:28px;font-size:13px;min-width:150px;cursor:pointer;';"
    "Array.from(orig.options).forEach(function(o){"
    "var op=document.createElement('option');"
    "op.value=o.value;op.text=o.text;"
    "if(o.selected)op.selected=true;"
    "neo.appendChild(op);"
    "});"
    "neo.addEventListener('change',function(e){"
    "orig.value=e.target.value;"
    "orig.dispatchEvent(new Event('change',{bubbles:true}));"
    "});"
    "orig.style.display='none';"
    "orig.parentNode.insertBefore(neo,orig);"
    "});"
    "}"
    "fix();setTimeout(fix,200);setTimeout(fix,600);"
    "})"
)

patched, n = re.subn(
    r'\.catch\(console\.error\)',
    move_js + ".catch(console.error)",
    html
)

if n == 0:
    print("⚠ Pattern non trouvé")
else:
    with open("viz2_regional.html", "w", encoding="utf-8") as f:
        f.write(patched)
    print(f"✓ Patch appliqué — rechargez viz2_regional.html")

✓ Patch appliqué — rechargez viz2_regional.html
